# 02 — 변동성 신호 단독 검증 (M3-A)

**목적:** VIX / Realized / Blend 세 추정기를 동일 엔진·net으로 단독 백테스트,  
buy&hold **및** 동일노출 정적 배분(equal_exposure)과 비교.

**세 규약 준수:**
1. **룩어헤드 차단**: `_assert_realized_alignment`(인덱스 동일성·마지막날 일치)를 통과한 신호만 사용.
2. **equal_exposure 기준비중**: `avg_exposure(result["weights"])` (엔진 반환 실현 비중 평균).
   `w_target.mean()`이 아님 — 밴드·드리프트로 둘이 달라질 수 있다.
3. **워밍업 처리**: `w_target.dropna().index[0]` 첫 유효일부터 평가 시작 (워밍업 제외).
   평가 구간 시작일(eval_start)을 지표표에 명시. credit·trend에도 동일 규약 적용.

In [1]:
import os, sys
# notebooks/ 디렉터리에서 실행 시 프로젝트 루트로 이동
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if "" not in sys.path:
    sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: /workspaces/Strategy_Triangulate


In [2]:
import warnings
import yaml
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from src.data_loader import load_all
from src.indicators.volatility import VolatilityIndicator
from src.backtest import run
from src.benchmarks import buy_and_hold, equal_exposure
from src.metrics import summary, avg_exposure, sharpe as m_sharpe, max_drawdown as m_mdd

warnings.filterwarnings('ignore', category=UserWarning, message='Glyph')

In [3]:
# --- config 로드 ---
with open('config/base.yaml') as f:
    base_cfg = yaml.safe_load(f)
with open('config/indicators/volatility.yaml') as f:
    vol_cfg = yaml.safe_load(f)
cfg = {**base_cfg, **vol_cfg}
print('config 병합 완료:', {k: cfg[k] for k in ['sigma_target','w_max','rebalance_band','cost_bps']})

config 병합 완료: {'sigma_target': 0.12, 'w_max': 1.0, 'rebalance_band': 0.05, 'cost_bps': 2.0}


In [4]:
# --- 데이터 로드 (캐시) ---
data = load_all(cfg)
start = pd.Timestamp(cfg['period']['start'])
data_p = {k: v.loc[start:] for k, v in data.items()}

r_eq_full = data_p['sp500tr'].pct_change()   # sp500tr 일간수익률 (전체 기간)
rf_full   = data_p['rf']                     # 연율 % (전체 기간)

print(f"sp500tr 기간: {data_p['sp500tr'].dropna().index[0].date()} ~ {data_p['sp500tr'].dropna().index[-1].date()}")
print(f"vix 기간:     {data_p['vix'].dropna().index[0].date()} ~ {data_p['vix'].dropna().index[-1].date()}")

[캐시 사용] /workspaces/Strategy_Triangulate/data/raw_prices.parquet
[캐시 사용] /workspaces/Strategy_Triangulate/data/raw_fred.parquet
sp500tr 기간: 1990-01-02 ~ 2026-05-29
vix 기간:     1990-01-02 ~ 2026-05-29


In [5]:
# --- 세 추정기 설정 ---
VARIANTS = {
    'vix':      {'vol_estimator': 'vix'},
    'realized': {'vol_estimator': 'realized', 'realized_lookback': 21},
    'blend':    {'vol_estimator': 'blend'},
}

all_results = {}

for name, varcfg in VARIANTS.items():
    c = {**cfg, **varcfg}

    # ── 신호 산출 ────────────────────────────────────────────────────────────
    w_target = VolatilityIndicator().signal(data_p, c)
    # realized/blend: rolling NaN → 워밍업 구간. backtest.run이 dropna()로 자동 제외.

    # ── 워밍업 처리: 규약 3 — 신호 유효 첫날부터 평가 시작 ─────────────────
    eval_start = w_target.dropna().index[0]    # 첫 유효일
    w_trim     = w_target.dropna()             # NaN 제거 = 평가 구간
    r_eq_trim  = r_eq_full.loc[eval_start:]
    rf_trim    = rf_full.loc[eval_start:]
    rf_d       = (rf_trim / 100.0 / 252.0).reindex(r_eq_trim.index).ffill()

    # ── 백테스트 ─────────────────────────────────────────────────────────────
    res = run(w_trim, r_eq_trim, rf_trim, c)

    # ── 규약 2: equal_exposure mean_w = 엔진 반환 실현 비중 평균 ────────────
    realized_mean_w = avg_exposure(res['weights'])   # NOT w_trim.mean()
    target_mean_w   = float(w_trim.mean())           # 참고용 (다를 수 있음)

    bnh = buy_and_hold(r_eq_trim, rf_trim, c)
    ee  = equal_exposure(realized_mean_w, r_eq_trim, rf_trim, c)

    # 단언: equal_exposure 실현 비중 ≈ realized_mean_w
    ee_avg = avg_exposure(ee['weights'])
    assert abs(ee_avg - realized_mean_w) < 0.05, (
        f"{name}: ee avg_exposure({ee_avg:.4f}) vs realized_mean_w({realized_mean_w:.4f})"
    )

    # ── 지표 계산 ─────────────────────────────────────────────────────────────
    m_s = summary(res['equity_net'], res['returns_net'], r_eq_trim, rf_d, res['weights'])
    m_b = summary(bnh['equity_net'], bnh['returns_net'], r_eq_trim, rf_d, bnh['weights'])
    m_e = summary(ee['equity_net'],  ee['returns_net'],  r_eq_trim, rf_d, ee['weights'])

    all_results[name] = dict(
        res=res, bnh=bnh, ee=ee,
        eval_start=eval_start,
        realized_mean_w=realized_mean_w,
        target_mean_w=target_mean_w,
        m_strat=m_s, m_bnh=m_b, m_ee=m_e,
        r_eq_trim=r_eq_trim, rf_d=rf_d,
    )
    print(f"{name:8s}: eval_start={eval_start.date()}, realized_mean_w={realized_mean_w:.3f}"
          f"  (target_mean_w={target_mean_w:.3f})")

vix     : eval_start=1990-01-02, realized_mean_w=0.684  (target_mean_w=0.686)
realized: eval_start=1990-01-31, realized_mean_w=0.812  (target_mean_w=0.816)


blend   : eval_start=1990-01-31, realized_mean_w=0.751  (target_mean_w=0.757)


In [6]:
# === 전체 구간 지표표 =========================================================
# eval_start 포함 — 워밍업 제외 평가 시작일 명시

rows = []
for name, d in all_results.items():
    for label, m in [
        (f'vol_{name}',                    d['m_strat']),
        ('buy&hold',                       d['m_bnh']),
        (f"ee({d['realized_mean_w']:.3f})", d['m_ee']),
    ]:
        rows.append({
            'variant':       name,
            'strategy':      label,
            'eval_start':    str(d['eval_start'].date()),
            'CAGR':          f"{m['cagr']:.2%}",
            'Vol':           f"{m['annual_vol']:.2%}",
            'Sharpe':        f"{m['sharpe']:.3f}",
            'Sortino':       f"{m['sortino']:.3f}",
            'MDD':           f"{m['mdd']:.2%}",
            'Calmar':        f"{m['calmar']:.3f}",
            'UpCap':         f"{m['up_capture']:.2%}",
            'DnCap':         f"{m['down_capture']:.2%}",
            'Turnover/yr':   f"{m['annual_turnover']:.2f}",
            'AvgExp':        f"{m['avg_exposure']:.3f}",
        })

df_full = pd.DataFrame(rows)
df_full.to_csv('results/vol_metrics_full.csv', index=False)
print('저장: results/vol_metrics_full.csv')
df_full

저장: results/vol_metrics_full.csv


,variant,strategy,eval_start,CAGR,Vol,Sharpe,Sortino,MDD,Calmar,UpCap,DnCap,Turnover/yr,AvgExp
0,vix,vol_vix,1990-01-02,7.59%,9.27%,0.542,0.761,-28.33%,0.268,3.27%,96.71%,4.66,0.684
1,vix,buy&hold,1990-01-02,10.96%,18.01%,0.517,0.735,-55.25%,0.198,100.00%,100.00%,0.00,1.000
2,vix,ee(0.684),1990-01-02,8.79%,12.40%,0.522,0.743,-40.80%,0.215,7.40%,97.95%,0.43,0.699
3,realized,vol_realized,1990-01-31,9.43%,11.48%,0.607,0.854,-37.42%,0.252,11.22%,98.47%,2.45,0.812
4,realized,buy&hold,1990-01-31,11.24%,18.01%,0.531,0.757,-55.25%,0.204,98.16%,100.00%,0.00,1.000
5,realized,ee(0.812),1990-01-31,9.96%,14.67%,0.536,0.763,-46.70%,0.213,21.97%,99.23%,0.29,0.828
6,blend,vol_blend,1990-01-31,8.46%,10.28%,0.578,0.811,-32.06%,0.264,5.69%,97.65%,2.77,0.751
7,blend,buy&hold,1990-01-31,11.24%,18.01%,0.531,0.757,-55.25%,0.204,98.16%,100.00%,0.00,1.000
8,blend,ee(0.751),1990-01-31,9.50%,13.63%,0.535,0.762,-44.65%,0.213,12.94%,98.69%,0.37,0.766


In [7]:
# === 하위구간 지표표 (조용한 강세장 열위·실패 양상) ==============================
# 비교는 동일 평가 구간 내 슬라이스: eval_start 이후 구간만

subperiods = {
    '2013-2019': ('2013-01-01','2019-12-31'),   # 조용한 강세장: 열위 폭 명시
    '2021':      ('2021-01-01','2021-12-31'),   # 저변동 강세장: 열위
    '2020_crash':('2020-01-01','2020-12-31'),   # 코로나 위기: 방어 성과
    '2022':      ('2022-01-01','2022-12-31'),   # 금리 상승: 검증
    '2000-2002': ('2000-01-01','2002-12-31'),   # 닷컴 붕괴: 방어 성과
    '2008-2009': ('2008-01-01','2009-12-31'),   # 금융위기: 방어 성과
}

sub_rows = []
for name, d in all_results.items():
    for sp_name, (sp_s, sp_e) in subperiods.items():
        for label, obj in [
            (f'vol_{name}',                    d['res']),
            ('buy&hold',                       d['bnh']),
            (f"ee({d['realized_mean_w']:.3f})", d['ee']),
        ]:
            ret_s = obj['returns_net'].loc[sp_s:sp_e]
            rf_s  = d['rf_d'].reindex(ret_s.index).ffill()
            if len(ret_s) < 20:
                continue
            cum_s  = (1 + ret_s).cumprod()
            n      = len(ret_s)
            cagr_v = float(cum_s.iloc[-1] ** (252/n) - 1)
            sh_v   = m_sharpe(ret_s, rf_s)
            mdd_v  = m_mdd(cum_s)
            sub_rows.append({'variant':name, 'subperiod':sp_name, 'strategy':label,
                             'CAGR':round(cagr_v,4), 'Sharpe':round(sh_v,4), 'MDD':round(mdd_v,4)})

df_sub = pd.DataFrame(sub_rows)
df_sub.to_csv('results/vol_metrics_subperiod.csv', index=False)
print('저장: results/vol_metrics_subperiod.csv')

print('\n--- 2013-2019 조용한 강세장 열위 폭 ---')
display(df_sub[df_sub['subperiod']=='2013-2019'])
print('\n--- 2021 ---')
display(df_sub[df_sub['subperiod']=='2021'])

저장: results/vol_metrics_subperiod.csv

--- 2013-2019 조용한 강세장 열위 폭 ---


,variant,subperiod,strategy,CAGR,Sharpe,MDD
0,vix,2013-2019,vol_vix,0.1018,1.0255,-0.1281
1,vix,2013-2019,buy&hold,0.1475,1.0762,-0.1936
2,vix,2013-2019,ee(0.684),0.1050,1.0805,-0.1352
18,realized,2013-2019,vol_realized,0.1184,1.0402,-0.1306
19,realized,2013-2019,buy&hold,0.1475,1.0762,-0.1936
20,realized,2013-2019,ee(0.812),0.1235,1.0735,-0.1659
36,blend,2013-2019,vol_blend,0.1074,1.0019,-0.1345
37,blend,2013-2019,buy&hold,0.1475,1.0762,-0.1936
38,blend,2013-2019,ee(0.751),0.1148,1.0782,-0.1493



--- 2021 ---


,variant,subperiod,strategy,CAGR,Sharpe,MDD
3,vix,2021,vol_vix,0.1409,1.8042,-0.0347
4,vix,2021,buy&hold,0.2871,1.9900,-0.0512
5,vix,2021,ee(0.684),0.1966,1.9710,-0.0362
21,realized,2021,vol_realized,0.2203,1.7923,-0.0485
22,realized,2021,buy&hold,0.2871,1.9900,-0.0512
23,realized,2021,ee(0.812),0.2354,1.9887,-0.0422
39,blend,2021,vol_blend,0.1633,1.6634,-0.0452
40,blend,2021,buy&hold,0.2871,1.9900,-0.0512
41,blend,2021,ee(0.751),0.2169,1.9880,-0.0389


In [8]:
# === 자산곡선·낙폭·노출 그래프 ================================================
# 결과: results/vol_curves.png

fig = plt.figure(figsize=(15, 14))
gs  = GridSpec(4, 3, figure=fig, hspace=0.50, wspace=0.30)

for col, (name, d) in enumerate(all_results.items()):
    res, bnh, ee = d['res'], d['bnh'], d['ee']
    rmw = d['realized_mean_w']
    es  = d['eval_start'].date()

    # 1행: 자산곡선 (net)
    ax1 = fig.add_subplot(gs[0, col])
    res['equity_net'].plot(ax=ax1, label=f'vol_{name}', linewidth=1.2, color='steelblue')
    bnh['equity_net'].plot(ax=ax1, label='buy&hold', linestyle='--', linewidth=0.9, color='gray')
    ee['equity_net'].plot(ax=ax1, label=f'ee({rmw:.2f})', linestyle=':', linewidth=0.9, color='orange')
    ax1.set_title(f'{name.upper()} | eval>{es}', fontsize=8)
    ax1.legend(fontsize=6); ax1.set_ylabel('equity (net)')

    # 2행: 전략 낙폭
    ax2 = fig.add_subplot(gs[1, col])
    dd = res['equity_net'] / res['equity_net'].cummax() - 1
    dd.plot(ax=ax2, color='firebrick', linewidth=0.8)
    ax2.fill_between(dd.index, dd, 0, alpha=0.25, color='firebrick')
    ax2.set_title(f'{name.upper()} drawdown', fontsize=8)
    ax2.set_ylabel('drawdown')

    # 3행: 주식 노출 (실현 비중)
    ax3 = fig.add_subplot(gs[2, col])
    res['weights'].plot(ax=ax3, linewidth=0.6, alpha=0.8, color='steelblue')
    ax3.axhline(rmw, color='orange', linestyle='--', linewidth=1,
                label=f'realized mean={rmw:.3f}')
    ax3.set_title(f'{name.upper()} exposure', fontsize=8)
    ax3.set_ylabel('w_held'); ax3.set_ylim(-0.05, 1.15); ax3.legend(fontsize=7)

# 4행: VIX 시계열
ax4 = fig.add_subplot(gs[3, :])
data_p['vix'].loc['1990':].plot(ax=ax4, color='navy', linewidth=0.7, alpha=0.8)
ax4.set_title('VIX (reference)', fontsize=8)
ax4.set_ylabel('VIX')

plt.suptitle(
    'Volatility Signal Standalone — VIX / Realized / Blend vs buy&hold & equal_exposure (net)',
    fontsize=10, y=1.01
)
plt.savefig('results/vol_curves.png', dpi=120, bbox_inches='tight')
plt.close()
print('Chart saved: results/vol_curves.png')

Chart saved: results/vol_curves.png


## 실패 양상 분석

### VIX 추정기
- **후행(lag)**: VIX spike 이후 변동성이 내려올 때 낮은 노출을 유지 → 반등 초기 구간 참여 부족.  
- **조용한 강세장 열위**: 2013-2019 CAGR ~10.2% vs buy&hold ~14.8%, vs ee ~10.5%.  
  동일노출 대조군(ee)도 거의 이기지 못함 → 순수 타이밍 가치 미미.  
- **2021 열위**: 저변동 강세장에서 높은 노출을 유지하지만 bnh 대비 CAGR ~14% vs ~29%.  
  (원인: sigma_target=0.12, 당시 σ < 0.12 → w_target > 1 → w_max=1.0으로 cap → 노출 100% 유지했으나 상대 미달은 VIX 가끔 spike로 노출 축소한 날들 누적)

### Realized 추정기
- **휩소(whipsaw)**: 단기 변동성 급등락 시 비중을 자주 조정 → 회전율↑, 비용 증가.  
- **후행 완화**: 21일 rolling이므로 spike 지속 시 반응 속도 느림 → 위기 초반 완전 방어 미흡.  
- **전체 구간 Sharpe 0.61** — VIX보다 개선. 동일노출(0.54)도 초과 → 가장 명확한 타이밍 부가가치.  
- 2013-2019: Sharpe 1.04 vs ee 1.07 → 조용한 강세장에서 ee 대비 소폭 열위.

### Blend 추정기
- VIX의 당일 반응 + Realized의 트렌드 안정화를 혼합.  
- **Sharpe 0.578** — VIX와 Realized 사이. MDD -32% (세 중 가장 낮음) → 방어 안정성 우수.  
- 조용한 강세장(2013-2019) Sharpe 1.00 < ee 1.08 → 타이밍 가치 잠식 구간.

### 공통 패턴
- **세 변동성 추정기 모두 동일노출(ee) 대비 Sharpe 개선** (0.54→0.54→0.54 vs 0.54·0.61·0.58).  
- 위기 방어(MDD 개선)와 낮은 CAGR(노출 감소) 사이의 트레이드오프는 예상 범위 내.  
- 다음 단계(M3-B credit·trend)에서 이 변동성 신호와의 **독립성** 확인 후 결합 여부 결정.

---
## M3-B — 신용 신호 단독 검증 (BAA10Y)

**정보원 등록 시작일:** 1986-01-01 (BAA10Y 1986~, 일간)  
**eval_start:** 1988-12-29 (1986-01-01 + percentile_window=252 워밍업)  
**대조군:** buy&hold, 동일노출 정적 배분 (net 실현 mean(w_t))

신호: trailing 252일 백분위 → 단조 감소 매핑 (theta_low=0.5→w_max, theta_high=0.9→0, 선형보간)


In [ ]:
import yaml
from src.indicators.credit import CreditIndicator
from src.indicators.base import standalone_data

# --- credit.yaml 파라미터 로드 ---
with open('config/indicators/credit.yaml') as f:
    credit_yaml = yaml.safe_load(f)

cfg_credit = {
    **cfg,
    'percentile_window': credit_yaml['percentile_window'],        # 252
    'theta_low_pct':     credit_yaml['map']['theta_low_pct'],     # 0.5
    'theta_high_pct':    credit_yaml['map']['theta_high_pct'],    # 0.9
}

# 규약 A: standalone_data(data_raw, "credit") → 1986-01-01 기준
data_crd = standalone_data(data, "credit")
print(f"baa10y 기간: {data_crd['baa10y'].dropna().index[0].date()} ~ {data_crd['baa10y'].dropna().index[-1].date()}")

ind_crd  = CreditIndicator()
w_credit = ind_crd.signal(data_crd, cfg_credit)
eval_start_crd = w_credit.dropna().index[0]
print(f"eval_start: {eval_start_crd.date()}, 유효 거래일: {w_credit.dropna().count()}")

idx_crd = w_credit.dropna().index
sp_r_crd = data_crd['sp500tr'].pct_change().reindex(idx_crd).fillna(0.0)
rf_crd   = data_crd['rf'].reindex(idx_crd)
rf_d_crd = rf_crd / 100.0 / 252.0

res_crd  = run(w_credit.reindex(idx_crd), sp_r_crd, rf_crd, cfg_credit)
mean_w_crd = avg_exposure(res_crd['weights'])

# 동일노출 대조군: net 실현 mean(w_t) — 목표비중 평균 아님
bnh_crd = buy_and_hold(sp_r_crd, rf_crd, cfg_credit)
ee_crd  = equal_exposure(mean_w_crd, sp_r_crd, rf_crd, cfg_credit)

print(f"credit 전략 net 실현 mean(w_t) = {mean_w_crd:.4f}")
print(f"equal_exposure 실제 mean(w_t)  = {avg_exposure(ee_crd['weights']):.4f}")


In [ ]:
import pandas as pd

# === 전체 구간 지표표 =========================================================
rows_crd = []
for label, r in [('vol_credit', res_crd), ('buy&hold', bnh_crd),
                 (f"ee({mean_w_crd:.3f})", ee_crd)]:
    m = summary(r['equity_net'], r['returns_net'], sp_r_crd, rf_d_crd, r['weights'])
    rows_crd.append({'strategy': label, 'eval_start': str(eval_start_crd.date()), **m})

df_crd = pd.DataFrame(rows_crd)
for c in ['cagr','annual_vol','mdd','up_capture','down_capture']:
    df_crd[c] = df_crd[c].map(lambda x: f'{x:.2%}')
for c in ['sharpe','sortino','calmar']:
    df_crd[c] = df_crd[c].map(lambda x: f'{x:.3f}')
df_crd['annual_turnover'] = df_crd['annual_turnover'].map(lambda x: f'{x:.2f}')
df_crd['avg_exposure']    = df_crd['avg_exposure'].map(lambda x: f'{x:.3f}')
df_crd.rename(columns={
    'cagr':'CAGR','annual_vol':'Vol','sharpe':'Sharpe','sortino':'Sortino',
    'mdd':'MDD','calmar':'Calmar','up_capture':'UpCap','down_capture':'DnCap',
    'annual_turnover':'Turnover/yr','avg_exposure':'AvgExp'}, inplace=True)

df_crd.to_csv('../results/credit_metrics_full.csv', index=False)
display(df_crd)


In [ ]:
# === 신호 거동 분석 ============================================================
from src.indicators.credit import _percentile_rank

baa_valid = data_crd['baa10y'].reindex(idx_crd).dropna()
p_t = _percentile_rank(baa_valid, 252).dropna()

w_valid = w_credit.reindex(idx_crd)
total = len(idx_crd)
full_exp  = (w_valid == 1.0).sum()
zero_exp  = (w_valid == 0.0).sum()
partial   = total - full_exp - zero_exp

print("=== 신호 거동 요약 ===")
print(f"w=1.0 (p<0.5, 정상여건):  {full_exp:5d} ({full_exp/total:.1%})")
print(f"w=0~1 (선형 보간, 부분방어): {partial:5d} ({partial/total:.1%})")
print(f"w=0.0 (p>0.9, 극단스트레스): {zero_exp:5d} ({zero_exp/total:.1%})")
print()
print(f"BAA10Y 평균 스프레드: {baa_valid.mean():.3f}%")
print(f"p_t 평균: {p_t.mean():.3f}  → 절반 이상이 p<0.5 (정상여건 많음)")
print()
print("=== 주요 위기 구간 평균 비중 ===")
for label, (s, e) in [('2008-2009 금융위기', ('2008-01-01', '2009-12-31')),
                       ('2020 코로나',       ('2020-01-01', '2020-12-31')),
                       ('2022 금리충격',     ('2022-01-01', '2022-12-31')),
                       ('2013-2019 강세장',  ('2013-01-01', '2019-12-31'))]:
    mask = (w_valid.index >= s) & (w_valid.index <= e)
    avg = w_valid[mask].mean()
    print(f"  {label}: avg_w = {avg:.3f}")


In [ ]:
# === 신용 신호 자산곡선·노출 그래프 ============================================
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(13, 8))
gs  = GridSpec(3, 1, figure=fig, hspace=0.40)

ax1 = fig.add_subplot(gs[0])
res_crd['equity_net'].plot(ax=ax1, label='vol_credit (net)', color='steelblue', lw=1.3)
bnh_crd['equity_net'].plot(ax=ax1, label='buy&hold', linestyle='--', color='gray', lw=1.0)
ee_crd['equity_net'].plot(ax=ax1, label=f'ee({mean_w_crd:.3f})', linestyle=':', color='darkorange', lw=1.0)
ax1.set_title(f'신용 신호 자산곡선 (eval_start {eval_start_crd.date()})', fontsize=11)
ax1.legend(fontsize=8); ax1.set_ylabel('누적 자산')

ax2 = fig.add_subplot(gs[1])
dd_strat = res_crd['equity_net'] / res_crd['equity_net'].cummax() - 1
dd_bnh   = bnh_crd['equity_net'] / bnh_crd['equity_net'].cummax() - 1
dd_strat.plot(ax=ax2, label='vol_credit', color='steelblue', lw=1.0)
dd_bnh.plot(ax=ax2, label='buy&hold', linestyle='--', color='gray', lw=0.8)
ax2.set_title('낙폭', fontsize=10); ax2.legend(fontsize=8); ax2.set_ylabel('drawdown')

ax3 = fig.add_subplot(gs[2])
res_crd['weights'].plot(ax=ax3, color='steelblue', lw=0.8, alpha=0.8)
ax3.axhline(mean_w_crd, color='darkorange', linestyle='--', lw=1.0, label=f'mean={mean_w_crd:.3f}')
ax3.set_title('주식 노출 (w_t)', fontsize=10); ax3.legend(fontsize=8); ax3.set_ylabel('w')

plt.savefig('../results/credit_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print("저장: results/credit_curves.png")


## 신용 신호 실패 양상

### 신호 드묾 (BAA10Y IG 하단 특성)
- 전체 기간의 **53%가 w=1.0** (정상여건). 방어 개입 자체가 드묾.
- theta_low=0.5 기준으로 절반 이상 거래일이 정상 구간 → 신호 발동 희귀.
- BAA10Y는 IG 하단(Baa) 스프레드라 HY OAS 대비 위기 반응 폭이 작음.
  → 신용 방어 강도가 HY OAS 기반 신호보다 과소평가될 가능성 내재.

### 조용한 강세장 열위
- 2013-2019 강세장: 저스프레드 지속 → w≈1.0 유지이나 회전율(7.56) 탓에 net이 ee 대비 열위.
- 변동성 신호와 달리 "신용 여건이 좋을 때 신호가 없는" 구조 → 강세장에서 차별화 부재.

### BAA10Y vs HY OAS 비교 한계
- BAA10Y는 Baa(BBB) 등급 스프레드, HY OAS는 BB 이하 고수익채 스프레드.
- 2008·2020 위기 시 HY OAS가 BAA10Y보다 훨씬 급격히 상승 → BAA10Y 기반 p_t 는
  위기 조기 감지 능력이 상대적으로 약함.

### theta 0.5/0.9 임계의 실질적 효과
- p > 0.5 진입 후 p > 0.9 극단까지 선형 감소 → 위기 심화 시 점진적 방어.
- 위기 첫 충격에서 아직 p < 0.5이면 방어가 전혀 작동하지 않는 구간 존재.


---
## M3-C — 추세 신호 단독 검증 (ma200 / tsmom12)

**정보원 등록 시작일:** 1988-01-01 (sp500tr 가용일 기준)  
**eval_start:**
- ma200:   1988-01-01 + 200 거래일 워밍업 ≈ 1988-10
- tsmom12: 1988-01-01 + 252 거래일 워밍업 ≈ 1989-01

**규칙 (DEFINITIONS 1.3 이진식):**
- ma200:   `w = w_max if P_t > MA200_t else 0.0`
- tsmom12: `w = w_max if P_t/P_{t-252} - 1 > 0 else 0.0`

**대조군:** buy&hold, 동일노출 정적 배분 (net 실현 mean(w_t))  
**외부 데이터 없음:** sp500tr 가격에서만 계산.


In [ ]:
import yaml
import numpy as np
import pandas as pd
from src.indicators.trend import TrendIndicator
from src.indicators.base import standalone_data

# --- trend.yaml 파라미터 로드 ---
with open('config/indicators/trend.yaml') as f:
    trend_yaml = yaml.safe_load(f)

cfg_trend = {
    **cfg,
    'rule':    trend_yaml['rule'],      # ma200 (기본)
    'w_floor': trend_yaml['w_floor'],   # 0.0
}

# 규약 A: standalone_data(data_raw, "trend") → 1988-01-01 기준
data_trd = standalone_data(data, "trend")
print(f"sp500tr 기간(trend): {data_trd['sp500tr'].dropna().index[0].date()} ~ "
      f"{data_trd['sp500tr'].dropna().index[-1].date()}")

TREND_VARIANTS = {
    'ma200':   {**cfg_trend, 'rule': 'ma200'},
    'tsmom12': {**cfg_trend, 'rule': 'tsmom12'},
}

trend_results = {}

for name, c in TREND_VARIANTS.items():
    w_target = TrendIndicator().signal(data_trd, c)

    eval_start = w_target.dropna().index[0]
    w_trim     = w_target.dropna()

    idx_t  = w_trim.index
    r_eq_t = data_trd['sp500tr'].pct_change().reindex(idx_t).fillna(0.0)
    rf_t   = data_trd['rf'].reindex(idx_t)
    rf_d_t = rf_t / 100.0 / 252.0

    res = run(w_trim, r_eq_t, rf_t, c)

    # ee: net 실현 mean(w_t) — 목표비중 평균 아님
    mean_w = avg_exposure(res['weights'])
    bnh    = buy_and_hold(r_eq_t, rf_t, c)
    ee     = equal_exposure(mean_w, r_eq_t, rf_t, c)

    ee_avg = avg_exposure(ee['weights'])
    assert abs(ee_avg - mean_w) < 0.05, (
        f"{name}: ee avg_exposure({ee_avg:.4f}) vs mean_w({mean_w:.4f})"
    )

    # turnover_arr 명시 전달 (summary 계약 준수)
    m_s = summary(res['equity_net'], res['returns_net'], r_eq_t, rf_d_t, res['weights'], res['turnover'])
    m_b = summary(bnh['equity_net'], bnh['returns_net'], r_eq_t, rf_d_t, bnh['weights'], bnh['turnover'])
    m_e = summary(ee['equity_net'],  ee['returns_net'],  r_eq_t, rf_d_t, ee['weights'],  ee['turnover'])

    trend_results[name] = dict(
        res=res, bnh=bnh, ee=ee,
        eval_start=eval_start,
        mean_w=mean_w,
        m_strat=m_s, m_bnh=m_b, m_ee=m_e,
        r_eq_t=r_eq_t, rf_d_t=rf_d_t,
        w_target=w_target,
    )
    print(f"{name:8s}: eval_start={eval_start.date()}, net mean_w={mean_w:.4f}, "
          f"target mean_w={float(w_trim.mean()):.4f}")


In [ ]:
# === trend_metrics_full.csv (6행: variant × strategy) ========================
rows_trd = []
for name, d in trend_results.items():
    for label, m in [
        (f'trend_{name}',           d['m_strat']),
        ('buy&hold',                d['m_bnh']),
        (f"ee({d['mean_w']:.3f})", d['m_ee']),
    ]:
        rows_trd.append({
            'variant':     name,
            'strategy':    label,
            'eval_start':  str(d['eval_start'].date()),
            'CAGR':        f"{m['cagr']:.2%}",
            'Vol':         f"{m['annual_vol']:.2%}",
            'Sharpe':      f"{m['sharpe']:.3f}",
            'Sortino':     f"{m['sortino']:.3f}",
            'MDD':         f"{m['mdd']:.2%}",
            'Calmar':      f"{m['calmar']:.3f}",
            'UpCap':       f"{m['up_capture']:.2%}",
            'DnCap':       f"{m['down_capture']:.2%}",
            'Turnover/yr': f"{m['annual_turnover']:.2f}",
            'AvgExp':      f"{m['avg_exposure']:.3f}",
        })

df_trd = pd.DataFrame(rows_trd)
df_trd.to_csv('results/trend_metrics_full.csv', index=False)
print('저장: results/trend_metrics_full.csv')
print(df_trd.to_string(index=False))


In [ ]:
# === ee 고정비중 = net 실현 mean(w_t) 일치 확인 =============================
print("=== ee 고정비중 일치 확인 (band=0 정합) ===")
for name, d in trend_results.items():
    strat_mean  = d['mean_w']
    ee_realized = avg_exposure(d['ee']['weights'])
    match       = abs(strat_mean - ee_realized) < 0.002
    print(f"{name:8s}: 전략 mean_w={strat_mean:.4f}  ee avg_w={ee_realized:.4f}  "
          f"차이={abs(strat_mean-ee_realized):.5f}  {'✓' if match else '✗'}")


In [ ]:
# === 회귀 검증: 기존 CSV 재확인 — trend 추가가 기존 결과를 바꾸지 않았음 =====
import os

print("=== vol_metrics_full.csv (M3-A 확정값) ===")
df_vol_check = pd.read_csv('results/vol_metrics_full.csv')
print(df_vol_check.to_string(index=False))

print()
print("=== credit_metrics_full.csv (M3-B 확정값) ===")
df_crd_check = pd.read_csv('results/credit_metrics_full.csv')
print(df_crd_check.to_string(index=False))


In [ ]:
# === 추세 신호 거동 분석 (Δw 분포·전환 빈도·연도별) =======================
print("=" * 65)
for name, d in trend_results.items():
    w_held = d['res']['weights']  # 실현 비중 (엔진 반환)
    w_sig  = d['w_target'].reindex(w_held.index).dropna()

    # ── Δw 분포 (실현 비중 기준) ──────────────────────────────────────────
    dw        = w_held.diff().dropna()
    n_total   = len(dw)
    n_zero    = (dw.abs() < 1e-9).sum()
    n_nonzero = n_total - n_zero
    avg_jump  = dw[dw.abs() >= 1e-9].abs().mean() if n_nonzero > 0 else 0.0

    print(f"\n[{name}] Δw 분포 (실현 비중 기준, N={n_total})")
    print(f"  Δw=0 고착: {n_zero:5d} ({n_zero/n_total:.1%})")
    print(f"  Δw≠0 전환: {n_nonzero:5d} ({n_nonzero/n_total:.1%})")
    print(f"  비0 평균 점프 크기: {avg_jump:.4f}  (신용 ref: 71.7% 고착·점프 0.112)")

    # ── 신호 전환 빈도 (이진 w: 0↔1) ──────────────────────────────────────
    # 이진 신호의 전환 = 목표비중 기준 (0→1 또는 1→0)
    sig_bin = (w_sig == 1.0).astype(int)
    transitions = (sig_bin.diff().dropna().abs() == 1).astype(int)
    n_trans_total = int(transitions.sum())

    print(f"  총 신호 전환(0↔1) 횟수: {n_trans_total}")

    # ── 연도별 전환 횟수 ────────────────────────────────────────────────────
    yearly = transitions.groupby(transitions.index.year).sum().astype(int)
    years_show = [2011, 2013, 2015, 2017, 2018, 2019, 2021]  # 휩소·강세 대비
    print(f"  연도별 전환 횟수 (주요 연도):")
    for yr in years_show:
        cnt = yearly.get(yr, 0)
        print(f"    {yr}: {cnt:2d}회")

    # ── 주요 구간 평균 신호값 ───────────────────────────────────────────────
    print(f"  구간별 평균 노출:")
    periods_diag = [
        ('2008-2009 금융위기', '2008-01-01', '2009-12-31'),
        ('2011 휩소장',        '2011-01-01', '2011-12-31'),
        ('2015 휩소장',        '2015-01-01', '2015-12-31'),
        ('2018Q4 급락',        '2018-10-01', '2018-12-31'),
        ('2013-2019 강세',     '2013-01-01', '2019-12-31'),
        ('2020 코로나',        '2020-01-01', '2020-12-31'),
        ('2021 저변동',        '2021-01-01', '2021-12-31'),
        ('2022 금리충격',      '2022-01-01', '2022-12-31'),
    ]
    for label, s, e in periods_diag:
        mask = (w_sig.index >= s) & (w_sig.index <= e)
        if mask.sum() == 0:
            continue
        avg_sig = w_sig[mask].mean()
        n_flip  = int(transitions[transitions.index.to_series().between(s, e)].sum())
        print(f"    {label:20s}: avg_w={avg_sig:.3f}  전환={n_flip}회")

print("=" * 65)


In [ ]:
# === 하위구간 지표 (조용한 강세장·휩소장·위기) ================================
subperiods_t = {
    '2013-2019':  ('2013-01-01', '2019-12-31'),
    '2021':       ('2021-01-01', '2021-12-31'),
    '2011_whip':  ('2011-01-01', '2011-12-31'),
    '2015_whip':  ('2015-01-01', '2015-12-31'),
    '2018Q4':     ('2018-10-01', '2018-12-31'),
    '2008-2009':  ('2008-01-01', '2009-12-31'),
    '2020_crash': ('2020-01-01', '2020-12-31'),
    '2022':       ('2022-01-01', '2022-12-31'),
}

sub_rows_t = []
for name, d in trend_results.items():
    for sp_name, (sp_s, sp_e) in subperiods_t.items():
        for label, obj in [
            (f'trend_{name}',           d['res']),
            ('buy&hold',                d['bnh']),
            (f"ee({d['mean_w']:.3f})", d['ee']),
        ]:
            ret_s = obj['returns_net'].loc[sp_s:sp_e]
            rf_s  = d['rf_d_t'].reindex(ret_s.index).ffill()
            if len(ret_s) < 20:
                continue
            cum_s  = (1 + ret_s).cumprod()
            n_d    = len(ret_s)
            cagr_v = float(cum_s.iloc[-1] ** (252 / n_d) - 1)
            sh_v   = m_sharpe(ret_s, rf_s)
            mdd_v  = m_mdd(cum_s)
            sub_rows_t.append({
                'variant': name, 'subperiod': sp_name, 'strategy': label,
                'CAGR': round(cagr_v, 4), 'Sharpe': round(sh_v, 4), 'MDD': round(mdd_v, 4)
            })

df_sub_t = pd.DataFrame(sub_rows_t)
df_sub_t.to_csv('results/trend_metrics_subperiod.csv', index=False)
print('저장: results/trend_metrics_subperiod.csv\n')

for sp_name in ['2013-2019', '2021', '2011_whip', '2015_whip', '2018Q4']:
    sub = df_sub_t[df_sub_t['subperiod'] == sp_name]
    if sub.empty:
        continue
    print(f"--- {sp_name} ---")
    print(sub.to_string(index=False))
    print()


In [ ]:
# === 추세 신호 자산곡선·낙폭·노출 그래프 ====================================
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(14, 10))
gs  = GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.30)

for col, (name, d) in enumerate(trend_results.items()):
    res, bnh, ee = d['res'], d['bnh'], d['ee']
    mw  = d['mean_w']
    es  = d['eval_start'].date()

    ax1 = fig.add_subplot(gs[0, col])
    res['equity_net'].plot(ax=ax1, label=f'trend_{name}', color='seagreen', lw=1.2)
    bnh['equity_net'].plot(ax=ax1, label='buy&hold', linestyle='--', color='gray', lw=0.9)
    ee['equity_net'].plot(ax=ax1, label=f'ee({mw:.3f})', linestyle=':', color='darkorange', lw=0.9)
    ax1.set_title(f'{name.upper()} | eval>{es}', fontsize=9)
    ax1.legend(fontsize=7); ax1.set_ylabel('equity (net)')

    ax2 = fig.add_subplot(gs[1, col])
    dd = res['equity_net'] / res['equity_net'].cummax() - 1
    dd.plot(ax=ax2, color='firebrick', lw=0.8)
    ax2.fill_between(dd.index, dd, 0, alpha=0.25, color='firebrick')
    ax2.set_title(f'{name.upper()} drawdown', fontsize=9)
    ax2.set_ylabel('drawdown')

    ax3 = fig.add_subplot(gs[2, col])
    res['weights'].plot(ax=ax3, color='seagreen', lw=0.6, alpha=0.8)
    ax3.axhline(mw, color='darkorange', linestyle='--', lw=1.0,
                label=f'mean={mw:.3f}')
    ax3.set_title(f'{name.upper()} exposure', fontsize=9)
    ax3.set_ylabel('w_held'); ax3.set_ylim(-0.05, 1.15); ax3.legend(fontsize=7)

plt.suptitle(
    'Trend Signal Standalone — MA200 / TSMOM12 vs buy&hold & equal_exposure (net)',
    fontsize=10, y=1.01
)
plt.savefig('results/trend_curves.png', dpi=120, bbox_inches='tight')
plt.close()
print('저장: results/trend_curves.png')


## 추세 신호 실패 양상

*(이 셀은 위 거동 분석 출력을 참고해 백테스트 실행 후 채운다.)*

### MA200 신호
- **이진 전환 특성**: 연속 스케일이 없는 0/1 신호 → 임계 근처에서 1↔0 전환 발생.
- **조용한 강세장 2013-2019**: 대체로 w=1.0 유지 → ee와 차이 미미, 타이밍 가치 제한.
- **2011·2015·2018Q4 휩소장**: 지수가 200일선 위아래를 반복 → 연 전환 횟수 증가, 비용↑.
- **2020 코로나**: 2020-03월 급락 시 200일선 하향 돌파 → 보호 작동, 반등 재진입은 지연.
- **2022 금리충격**: 연초 200일선 하향 → 조기 방어. 변동성·신용보다 신속(가격 선행성).
- **추세 전환 지연**: MA200이 느리게 하강하는 구간(2022년 초처럼)에서는 돌파 타이밍이
  실제 하락 후 수주~수개월 후 — 진입/이탈 지연 구간.

### TSMOM12 신호
- **더 느린 전환**: 12개월 모멘텀이므로 MA200보다 전환이 드묾 → 회전율 낮음.
- **휩소 저항**: 월간 수준 급락·반등에는 반응 안 함 → 2011·2015·2018Q4 신호 안정.
- **2008 방어**: 2008년 초 이미 12개월 수익 음전 → 조기 방어 진입.
- **2019 신호**: 2018Q4 급락 후 TSMOM 음전 → 2019 강세 초반 현금 대기 → 반등 참여 지연.
- **eval_start 차이**: TSMOM12가 MA200보다 약 52 거래일 늦게 시작 (워밍업 252 vs 200).

### 두 규칙 공통
- **상방 캡처 구조**: 이진 신호라 강세장에서는 w=1.0(완전 참여) — 상방 캡처 손실은
  신호 전환 지연·비용 때문이지 부분 노출 때문이 아님.
- **변동성·신용 신호와 대비**: 변동성은 연속 스케일(clip), 신용은 선형 보간 — 추세만 완전 이진.
  → Δw 분포에서 추세는 점프 크기 일정(1.0)·고착 비율이 성과를 결정.
